# ETF Explorer (v1)

Discovery view over Stashaway's full ETF Explorer offering (~98 ETFs).
Computes multi-window metrics (1Y/3Y/5Y) + correlation with your combined book.
Renders to `reports/etf_explorer.html`.

In [ ]:
from datetime import date
from pathlib import Path

from hailmary.allocation.book_config import MGMT_FEES_ANNUAL, ROLES
from hailmary.allocation.etf_explorer import build_etf_explorer, render_etf_explorer_report
from hailmary.allocation.portfolios import from_parsed
from hailmary.allocation.statements import parse_statement
from hailmary.allocation.returns import last_business_day_on_or_before
from hailmary.data.providers import YahooFinanceProvider

ETF_XLSX = Path('../../data/stashaway_etf_universe.xlsx')
STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
REPORT_PATH = Path('../../reports/etf_explorer.html')
START = date(2020, 1, 1)
END = last_business_day_on_or_before(date.today())
TARGET_ANN_RETURN = 0.05
print(f'window: {START}..{END}')

## Load user's book (for correlation reference)

In [ ]:
parsed = parse_statement(STATEMENT_PATH)
portfolios = [
    from_parsed(
        p,
        roles=ROLES[p.name],
        metadata={'management_fee_annual': MGMT_FEES_ANNUAL.get(p.name, 0.0)},
    )
    for p in parsed if p.name in ROLES
]
provider = YahooFinanceProvider()
fx_bars = provider.get_bars(['USDSGD=X'], START, END)
fx_series_usd_sgd = fx_bars.xs('USDSGD=X', level=0)['close']
print(f'{len(portfolios)} portfolios loaded')

## Build the explorer DataFrame

In [ ]:
explorer_df = build_etf_explorer(
    ETF_XLSX,
    portfolios=portfolios,
    price_source=provider,
    fx_series_usd_sgd=fx_series_usd_sgd,
    start=START,
    end=END,
)
print(f'{len(explorer_df)} ETFs, {explorer_df["has_data"].sum()} with Yahoo data')
explorer_df.head(20)

## Render HTML report

In [ ]:
out = render_etf_explorer_report(
    explorer_df,
    REPORT_PATH,
    target_ann_return=TARGET_ANN_RETURN,
)
print(f'Wrote {out.resolve()}')